# 通常実験：全件データの学習・4方式比較

まずテスト用 colab_smoke.ipynb を完了してください。通常用は全件を読み込み、長期学習には明示確認が必要です。
セル4AまではCPUランタイムで実行できます。4Bと学習・比較はGPUを使用します。設定は最初のフォームに集約しています。
通常はtt2・s2ut・unit（このコーパスのtrain音声で学習するボコーダー）を学習して比較をONにします。TT2のHiFi-GAN代替モードではmelも必要です。S2UTのpaperモードは公式LJSpeech重みを使う代替です。
通常はpaper_exact（TT2実効batch1024）。重い場合はpaper_practicalを明示選択します。16kHz/80-bin target Melは維持するため、完全な論文再現ではありません。
保存先は training/実験名。切断後も同じ設定で再実行します。コーパスは変更しません。

CPUのまま4A → 4A.5（音声ZIP準備）を完了してからGPUへ切り替えてください。同じ設定でセル1を再実行し、4Bまたは5へ進めます。Drive接続・コード取得・ローカル環境は自動復元します。GPUの変更自体はColabで手動操作してください。


In [ ]:
#@title 1. 設定（変更するのは基本ここだけ）
CORPUS_PATH = '/content/drive/MyDrive/ja-en-direct-s2st-corpus-data' #@param {type:"string"}
OUTPUT_BASE = '/content/drive/MyDrive/ja-en-direct-s2st-benchmark-data' #@param {type:"string"}
EXPERIMENT = 'experiment-001' #@param {type:"string"}
TRAIN_TARGET = 'tt2' #@param ["tt2", "s2ut", "unit", "mel"]
RUN_TRAINING = False #@param {type:"boolean"}
RUN_BENCHMARK = False #@param {type:"boolean"}
CONFIRM_FULL_DATA = True #@param {type:"boolean"}
TOTAL_UPDATES = 10000 #@param {type:"integer"}
S2UT_TOTAL_UPDATES = 400000 #@param {type:"integer"}
CONFIRM_TRAINING = False #@param {type:"boolean"}
SESSION_MINUTES = 60 #@param {type:"integer"}
# 4A/4B専用。並列数は上限で、実際の数は負荷に応じて調整します。
PREP_MAX_WORKERS = 32 #@param {type:"integer"}
HUBERT_MAX_BATCH = 32 #@param {type:"integer"}
RECHECK_AUDIO = False #@param {type:"boolean"}
# 以下は自動設定。テストと通常実験で保存先を共有しません。
MODE = 'training'
PROFILE = 'pilot'
DATA_LIMIT = None
MODEL_SIZE = 'fisher'
REPRODUCTION_MODE = 'paper_exact' #@param ["paper_exact", "paper_practical"]
TT2_VOCODER_MODE = 'griffin_lim' #@param ["griffin_lim", "hifigan"]
# trained: このコーパスのtrain音声で学習。paper: 公式LJSpeech配布重みの代替（原論文と学習データが異なる）。
S2UT_VOCODER_MODE = 'trained' #@param ["trained", "paper"]
TT2_BATCH_SIZE = 8 #@param {type:"integer"}
PRACTICAL_UPDATE_FREQ = 1 #@param {type:"integer"}
# exact: TT2実効batch1024、S2UT公式max_tokens20000/update_freq4（追加分割禁止）。practicalのみ変更可。
def training_run_name(kind):
    base = kind + '-' + MODE + '-' + EXPERIMENT
    return base + '-' + REPRODUCTION_MODE + ('-recipe-v3' if kind == 's2ut' else '-recipe-v2') if kind in ('tt2', 's2ut') else base
PERFORMANCE = 'gpu80'
# セル5。分割・精度変更は別の実験条件として扱います。
TRAIN_OPTIMIZE = True #@param {type:"boolean"}
TRAIN_MAX_WORKERS = 8 #@param {type:"integer"}
TRAIN_CACHE_GB = 8 #@param {type:"number"}
TRAIN_SAVE_INTERVAL = 1 #@param {type:"integer"}
TRAIN_BACKUP_MIN_SECONDS = 600 #@param {type:"number"}
TRAIN_BACKUP_MAX_GB = 20 #@param {type:"number"}
TRAIN_ADAPTIVE_BATCH = False #@param {type:"boolean"}
# 本学習前に別プロセスで測定し、選択後は固定。試運転の重みは捨てます。
TRAIN_CALIBRATE_BATCH = True #@param {type:"boolean"}
# S2UT paper_exactでは容量確認のみ。同時処理数を増減・追加分割しません。
TRAIN_CALIBRATION_MAX_BATCH = 1024 #@param {type:"integer"}
TRAIN_VRAM_RESERVE_RATIO = 0.10 #@param {type:"number"}
TRAIN_CALIBRATION_STEPS = 3 #@param {type:"integer"}
TRAIN_CALIBRATION_OBJECTIVE = 'throughput' #@param ["throughput", "capacity"]
TRAIN_PRECISION = 'default' #@param ["default", "fp32", "bf16"]
CHUNK_UPDATES = 100

# GPU切り替え後もセル1＋目的のセルだけで復旧できます。
def ensure_workspace():
    global CORPUS, PERSISTENT, DATA, REPO, run, _WORKSPACE_KEY, os, subprocess, json, Path
    from pathlib import Path
    key = (CORPUS_PATH, OUTPUT_BASE, MODE, EXPERIMENT)
    if globals().get('_WORKSPACE_KEY') == key and Path('/content/ja-en-direct-s2st-benchmark/.git').exists() and Path('/content/drive/MyDrive').is_dir():
        return
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    import os, subprocess, json
    # 自分の保存先へ変更。コーパスの下に出力を置かないこと。
    CORPUS = Path(CORPUS_PATH)
    PERSISTENT = Path(OUTPUT_BASE) / MODE / EXPERIMENT
    DATA = PERSISTENT / 'data'
    REPO = Path('/content/ja-en-direct-s2st-benchmark')
    assert EXPERIMENT and all(c.isalnum() or c in '-_' for c in EXPERIMENT)
    assert not PERSISTENT.resolve().is_relative_to(CORPUS.resolve())
    PERSISTENT.mkdir(parents=True, exist_ok=True)
    def run(*args):
        command = [str(a) for a in args]
        log = PERSISTENT / 'session.log'
        print('実行:', ' '.join(command), flush=True)
        with log.open('a', encoding='utf-8') as handle:
            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                       text=True, encoding='utf-8', errors='replace', bufsize=1)
            for line in process.stdout:
                print(line, end='', flush=True)
                handle.write(line)
                handle.flush()
            status = process.wait()
        if status:
            raise RuntimeError(f'終了コード {status}。直前のエラーと {log} を確認してください')
    assert not PERSISTENT.resolve().is_relative_to(CORPUS.resolve())
    revision_file = PERSISTENT / 'repository-revision.txt'
    if not REPO.exists():
        run('git', 'clone', '--recurse-submodules',
            'https://github.com/Yaaamashiro/ja-en-direct-s2st-benchmark.git', REPO)
    if revision_file.exists():
        revision = revision_file.read_text().strip()
        run('git', '-C', REPO, 'fetch', 'origin', revision)
        run('git', '-C', REPO, 'checkout', '--detach', revision)
    else:
        revision = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
        revision_file.write_text(revision)
    os.chdir(REPO)
    print('Pinned repository:', revision)
    os.environ['CORPUS_ROOT'] = str(CORPUS)
    _WORKSPACE_KEY = key

def ensure_runtime(require_gpu=False):
    global PYTHON
    import runpy
    ensure_workspace()
    helper = REPO / 'scripts/colab/runtime.py'
    if not helper.is_file():
        raise RuntimeError('保存済みrevisionが旧版です。この機能を使うには新コードと新しい実験名を使用してください。旧データは削除しないでください。')
    PYTHON = runpy.run_path(str(helper))['ensure_runtime'](REPO, PERSISTENT, run, require_gpu=require_gpu)

def cli(*args):
    run(PYTHON, '-m', 'direct_s2st.cli', *args)

def configure_preparation():
    import os
    os.environ.update(S2ST_DRIVE_SAFE='1', S2ST_BACKUP_MIN_SECONDS=str(TRAIN_BACKUP_MIN_SECONDS), S2ST_BACKUP_MAX_GB=str(TRAIN_BACKUP_MAX_GB))
    assert 1 <= PREP_MAX_WORKERS <= 32
    assert 1 <= HUBERT_MAX_BATCH <= 32
    os.environ.update(S2ST_PREP_WORKERS=str(PREP_MAX_WORKERS),
                      S2ST_PREP_ADAPTIVE='1', S2ST_HUBERT_BATCH_MAX=str(HUBERT_MAX_BATCH),
                      S2ST_PREP_RECHECK='1' if RECHECK_AUDIO else '0')


In [ ]:
#@title 2. Drive接続・コード取得（以後のセルからも自動実行）
ensure_workspace()


In [ ]:
#@title 3. ランタイム構築（CPU可・以後のセルからも自動実行）
ensure_runtime(require_gpu=False)


In [ ]:
#@title 4A. CPUデータ準備（GPU不要）
assert CONFIRM_FULL_DATA, '通常用は全件を処理します。設定のCONFIRM_FULL_DATAを確認して有効にしてください'
ensure_runtime(require_gpu=False)
configure_preparation()
limit_args = [] if DATA_LIMIT is None else ['--limit', str(DATA_LIMIT)]
run(PYTHON, '-m', 'direct_s2st.preparation_workflow', '--stage', '4a',
    '--profile', PROFILE, *limit_args)
print('CPU準備完了:', DATA)
print('CPUのままセル4A.5（音声ZIP準備）を実行してください。')


In [ ]:
#@title 4A.5. 音声ZIP準備（CPU・中断再開可）
assert CONFIRM_FULL_DATA, '通常用は全件を処理します。CONFIRM_FULL_DATAを有効にしてください'
ensure_runtime(require_gpu=False)
configure_preparation()
run(PYTHON, '-m', 'direct_s2st.preparation_workflow', '--stage', 'audio-packs',
    '--profile', PROFILE)
print('完成ZIPはDriveに保存済みです。GPUへ切り替えて同じ設定でセル1 → 4Bへ進んでください。')


In [ ]:
#@title 4B. HuBERT Unit抽出・S2UT準備（先にCPUセル4A.5を完了）
assert CONFIRM_FULL_DATA, '通常用は全件を処理します。CONFIRM_FULL_DATAを有効にしてください'
ensure_runtime(require_gpu=True)
configure_preparation()
assert (DATA / 'common/dataset-lock.json').is_file(), '先にセル4Aを完了してください'
limit_args = [] if DATA_LIMIT is None else ['--limit', str(DATA_LIMIT)]
run(PYTHON, '-m', 'direct_s2st.preparation_workflow', '--stage', '4b',
    '--profile', PROFILE, *limit_args)
print('Unit準備完了:', DATA)


In [ ]:
#@title 5. 選択したモデルを学習・再開
# tt2 / s2ut / unit / melをそれぞれ選んで実行。完了済みsnapshotがあれば自動再開します。
if RUN_TRAINING:
    selected_updates = S2UT_TOTAL_UPDATES if TRAIN_TARGET == 's2ut' else TOTAL_UPDATES
    assert selected_updates > 0 and SESSION_MINUTES > 0
    assert selected_updates <= 10 or CONFIRM_TRAINING, '長期学習はCONFIRM_TRAINING=Trueが必要です'
    ensure_runtime(require_gpu=True)
    configure_preparation()
    BACKEND = TRAIN_TARGET
    if TRAIN_TARGET == 's2ut':
        run(PYTHON, '-m', 'direct_s2st.s2ut.migrate_labels', '--common-root', DATA / 'common', '--data-root', DATA / 's2ut/fairseq')
    assert (REPO / 'src/direct_s2st/recipes.py').is_file(), '保存済みrevisionが旧版です。docs/PAPER_RECIPES.mdの明示移行手順を確認してください。データは削除不要です。'
    assert not TRAIN_CALIBRATE_BATCH or (REPO / 'src/direct_s2st/batch_calibration.py').is_file(), '保存済みrevisionにVRAM事前測定がありません。新コードへの明示移行が必要です。準備データは削除不要です。'
    RUN_NAME = training_run_name(TRAIN_TARGET)
    assert 0 <= TRAIN_MAX_WORKERS <= 32
    options = dict(performance=PERFORMANCE, optimize=TRAIN_OPTIMIZE,
                   num_workers=TRAIN_MAX_WORKERS, cache_gb=TRAIN_CACHE_GB,
                   save_interval=TRAIN_SAVE_INTERVAL,
                   adaptive_batch=TRAIN_ADAPTIVE_BATCH, precision=TRAIN_PRECISION,
                   calibrate_batch=TRAIN_CALIBRATE_BATCH,
                   calibration_max_batch=TRAIN_CALIBRATION_MAX_BATCH,
                   calibration_reserve_ratio=TRAIN_VRAM_RESERVE_RATIO,
                   calibration_steps=TRAIN_CALIBRATION_STEPS,
                   calibration_objective=TRAIN_CALIBRATION_OBJECTIVE,
                   reproduction_mode=REPRODUCTION_MODE,
                   tt2_vocoder_mode=TT2_VOCODER_MODE, s2ut_vocoder_mode=S2UT_VOCODER_MODE)
    if TRAIN_TARGET == 'tt2':
        options['batch_size'] = TT2_BATCH_SIZE
    if TRAIN_TARGET in ('tt2', 's2ut') and REPRODUCTION_MODE == 'paper_practical':
        options['update_freq'] = PRACTICAL_UPDATE_FREQ
    assert RUN_NAME and all(c.isalnum() or c in '-_' for c in RUN_NAME)
    CONFIG = PERSISTENT / (RUN_NAME + '.json')
    # 設定生成も固定Python環境で実行する。
    builder = "import json, sys\nfrom pathlib import Path\nfrom direct_s2st.colab import make_config\ncfg = make_config(sys.argv[1], sys.argv[2], '/content/s2st-runtime/environment.json', sys.argv[4], sys.argv[5], **json.loads(sys.argv[6]))\npath = Path(sys.argv[3])\nif path.exists():\n    assert json.loads(path.read_text()) == cfg, 'Config changed: use a new run name'\nelse:\n    path.write_text(json.dumps(cfg, indent=2))\nprint(json.dumps(cfg, indent=2))\n"
    run(PYTHON, '-c', builder, REPO, DATA, CONFIG, BACKEND, MODEL_SIZE, json.dumps(options))
    args = [PYTHON, '-m', 'direct_s2st.colab', '--config', CONFIG,
            '--work-root', '/content/s2st-work/' + RUN_NAME,
            '--backup-root', PERSISTENT / 'checkpoints' / RUN_NAME,
            '--total-updates', str(selected_updates), '--chunk-updates', str(CHUNK_UPDATES),
            '--session-seconds', str(SESSION_MINUTES * 60), '--auto-resume']
    if CONFIRM_TRAINING:
        args.append('--confirm-training')
    run(*args)
else:
    print('学習はOFFです。設定のRUN_TRAININGを有効にすると開始します。')


## 学習後の4方式比較

通常設定ではTT2・S2UT・Unit vocoderの3つの学習runが必要です。S2UT_VOCODER_MODE=paperはLJSpeech配布重みの代替で、unit学習を省略できますが論文の学習データ条件とは異なります。テストの2更新では正常な音声が出る保証はありません。
RUN_BENCHMARKをONにして以下を実行します。結果はDriveに保存します。
比較レポートを作り直す場合だけ、次のセルのOVERWRITE_COMPARISONをTrueにしてください。
BLASERは既定で無効です。別途依存確認が必要です。


In [ ]:
#@title 6. 学習済みモデルの復元・比較準備
if RUN_BENCHMARK:
    ensure_runtime(require_gpu=True)
    RUN_DIRECT_MODELS = RUN_BENCHMARK
    assert (REPO / 'src/direct_s2st/recipes.py').is_file(), '保存済みrevisionが旧版です。docs/PAPER_RECIPES.mdの明示移行手順を確認してください。データは削除不要です。'
    RUN_CASCADE = RUN_BENCHMARK
    RUN_S2T_TTS = RUN_BENCHMARK
    RUN_EVALUATION = RUN_BENCHMARK
    RUN_COMPARISON = RUN_BENCHMARK
    INFERENCE_LIMIT = DATA_LIMIT
    RESUME_INFERENCE = True
    OVERWRITE_COMPARISON = False
    # 学習済み・復元済みrunを指定。方式とvocoderごとに別runで学習します。
    TRAINED_RUNS = {
        's2ut': '/content/s2st-work/' + training_run_name('s2ut'),
        'translatotron2': '/content/s2st-work/' + training_run_name('tt2'),
    }
    VOCODER_ROOTS = {
        'unit': '/content/s2st-work/unit-' + MODE + '-' + EXPERIMENT,
        'mel': '/content/s2st-work/mel-' + MODE + '-' + EXPERIMENT,
    }
    RESTORE_TRAINING_RUNS = [
        training_run_name(kind) for kind in ('s2ut', 'tt2', 'unit', 'mel')
        if not (kind == 'unit' and S2UT_VOCODER_MODE == 'paper')
        and not (kind == 'mel' and TT2_VOCODER_MODE == 'griffin_lim')
        and not (Path('/content/s2st-work') / training_run_name(kind) /
                ('generator.pt' if kind in ('unit', 'mel') else 'checkpoints/checkpoint_last.pt')).exists()
    ]
    
    for name in RESTORE_TRAINING_RUNS:
        assert name and all(c.isalnum() or c in '-_' for c in name)
        run(PYTHON, '-m', 'direct_s2st.colab',
            '--config', PERSISTENT / (name + '.json'),
            '--work-root', '/content/s2st-work/' + name,
            '--backup-root', PERSISTENT / 'checkpoints' / name,
            '--resume', '--restore-only')
    cli('corpus', 'validate', '--profile', PROFILE)
    test_rows = [json.loads(line) for line in (DATA / 'common/test.jsonl').read_text().splitlines() if line.strip()]
    assert test_rows, 'common/test is empty'
    if RUN_DIRECT_MODELS or RUN_COMPARISON:
        assert INFERENCE_LIMIT is None or INFERENCE_LIMIT >= len(test_rows), (
            'Four-system acceptance requires all common/test IDs. Prepare a separate tiny subset for smoke, or use None.'
        )
    RUNS = Path(os.environ['RUNS_ROOT'])
    COMPARISON_NAME = EXPERIMENT + '-' + REPRODUCTION_MODE + '-' + TT2_VOCODER_MODE + '-' + S2UT_VOCODER_MODE
    COMPARE_CONFIG = PERSISTENT / 'comparison-configs' / COMPARISON_NAME
    prepare_comparison = "import json, sys\nfrom direct_s2st.colab_compare import stage_direct_artifacts, build_configs, save_configs\nrepo, runs, name, out = sys.argv[1:5]\nmodes = dict(tt2_vocoder_mode=sys.argv[8], s2ut_vocoder_mode=sys.argv[9])\nif sys.argv[5] == 'True':\n    stage_direct_artifacts(runs, name, json.loads(sys.argv[6]), json.loads(sys.argv[7]), **modes)\nsave_configs(build_configs(repo, runs, name, **modes), out)\n"
    run(PYTHON, '-c', prepare_comparison, REPO, RUNS, COMPARISON_NAME, COMPARE_CONFIG,
        str(RUN_DIRECT_MODELS), json.dumps(TRAINED_RUNS), json.dumps(VOCODER_ROOTS),
        TT2_VOCODER_MODE, S2UT_VOCODER_MODE)
    def configured(command, config_name, *extra):
        args = [*command, '--config', COMPARE_CONFIG / (config_name + '.yaml'), '--profile', PROFILE, *extra]
        cli(*args)
    resume_flags = ['--resume'] if RESUME_INFERENCE else []
    limit_flags = [] if INFERENCE_LIMIT is None else ['--limit', str(INFERENCE_LIMIT)]
else:
    print('比較はOFFです。選択したモードで必要な学習済みrunがそろったらRUN_BENCHMARKを有効にしてください。')


In [ ]:
#@title 7. 英語音声を生成（4方式）
if RUN_BENCHMARK:
    if RUN_DIRECT_MODELS:
        for system, kind in [('s2ut', 'unit'), ('translatotron2', 'mel')]:
            configured([system, 'infer'], system, *resume_flags)
            configured(['vocoder', kind, 'infer'], kind, *resume_flags)
    if RUN_CASCADE:
        configured(['cascade', 'run'], 'cascade', '--split', 'test', *limit_flags, *resume_flags)
    if RUN_S2T_TTS:
        configured(['s2t-tts', 'run'], 's2t_tts', '--split', 'test', *limit_flags, '--dry-run')
        configured(['s2t-tts', 'run'], 's2t_tts', '--split', 'test', *limit_flags, *resume_flags)
else:
    print('比較はOFFです。選択したモードで必要な学習済みrunがそろったらRUN_BENCHMARKを有効にしてください。')


In [ ]:
#@title 8. 共通評価・比較レポート
if RUN_BENCHMARK:
    if RUN_EVALUATION:
        for system in ('s2ut', 'translatotron2', 'cascade', 's2t_tts'):
            configured(['evaluate', 'run'], system + '-eval', *resume_flags)
    if RUN_COMPARISON:
        comparison_flags = ['--overwrite'] if OVERWRITE_COMPARISON else ['--resume']
        configured(['evaluate', 'verify'], 'comparison', *comparison_flags)
        configured(['evaluate', 'aggregate'], 'comparison', *comparison_flags)
        print('Drive results:', DATA / 'results')
        print((DATA / 'results/report.md').read_text())
else:
    print('比較はOFFです。選択したモードで必要な学習済みrunがそろったらRUN_BENCHMARKを有効にしてください。')


出力: Driveの実験フォルダ内 runs/ に予測音声と評価、data/results/ に比較レポート、checkpoints/ に学習復元点、session.log に実行ログ。
通常用とテスト用のデータ・checkpoint・結果は混ぜません。実験条件を変える場合は新しい実験名を使ってください。
